# Instagram Profile Analytics with Scavio

Turn any public Instagram profile into an analytics snapshot -- followers, recent-post engagement rate, and content themes -- with the Scavio **Python SDK** and an LLM. A free alternative to HypeAuditor-style profile reports.

We fetch with the SDK, compute the metrics in plain Python, then hand a compact summary to the LLM. That keeps large Instagram payloads out of the model's context.

**What you will learn:**
- Pull a profile with `client.instagram.profile`
- Pull recent posts with `client.instagram.user_posts`
- Compute an engagement rate and summarize themes with an LLM

In [1]:
# pip install scavio langchain-openai python-dotenv

In [2]:
from dotenv import load_dotenv
from scavio import ScavioClient
from langchain_openai import ChatOpenAI

load_dotenv(override=True)
client = ScavioClient()  # reads SCAVIO_API_KEY

In [3]:
HANDLE = "natgeo"

profile = client.instagram.profile(username=HANDLE).get("data", {})
followers = profile.get("follower_count", 0)
print(f"@{profile.get('username')} - {followers:,} followers, "
      f"{profile.get('media_count')} posts, category={profile.get('category')}")

@natgeo - 269,202,442 followers, 31746 posts, category=


In [4]:
# Pull recent posts and compute engagement.
items = (client.instagram.user_posts(username=HANDLE, count=12).get("data", {})).get("items", [])
posts = [
    {
        "likes": p.get("like_count", 0),
        "comments": p.get("comment_count", 0),
        "caption": (p.get("caption_text") or "")[:160],
    }
    for p in items
]
n = len(posts) or 1
avg_eng = sum(p["likes"] + p["comments"] for p in posts) / n
eng_rate = (avg_eng / followers * 100) if followers else 0
print(f"Recent posts: {len(posts)}")
print(f"Avg engagement/post: {avg_eng:,.0f}")
print(f"Engagement rate: {eng_rate:.2f}%")

Recent posts: 12
Avg engagement/post: 66,382
Engagement rate: 0.02%


In [5]:
# Summarize content themes from the captions.
captions = "\n".join(f"- ({p['likes']:,} likes) {p['caption']}" for p in posts)
prompt = (
    f"Instagram account @{HANDLE} ({followers:,} followers, {eng_rate:.2f}% engagement).\n"
    f"Recent post captions and likes:\n{captions}\n\n"
    "In 4-5 bullets: what does this account post about, what format/theme drives "
    "the most engagement, and one recommendation. Use only the data shown."
)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print(llm.invoke(prompt).content)

- The @natgeo account primarily posts about nature, wildlife, historical events, and cultural experiences, often featuring captivating visuals and storytelling elements.
- Posts that highlight engaging animal behavior or unique wildlife, such as meerkats and green sea turtles, tend to drive the most engagement, as seen in the likes received.
- Content related to historical events, like the eruption of Mount Vesuvius and its impact on Pompeii, also garners significant interest, though not as high as animal-related posts.
- A recommendation would be to increase the frequency of posts featuring compelling animal stories or behaviors, as these seem to resonate more with the audience and drive higher engagement.
